# Smoke test on Kaggle (yolo11n, 3 epochs, imgsz=320)

Run this once before a real training session to verify the end-to-end flow.
Takes ~5 min on T4. Verifies: GPU, git clone, install, dataset unzip + sha256,
training, run_meta.json write, test-split eval.

**Prereqs:** dataset Kaggle Dataset attached via 'Add Data', Internet ON, GPU ON.
Set `INPUT_DATASET_DIR` to where the dataset zip lives under `/kaggle/input/`.

In [ ]:
REPO_URL    = 'https://github.com/tahmid013/yolo.git'
REPO_BRANCH = 'main'
INPUT_DATASET_DIR = '/kaggle/input/safety-equipment-yolo-v1'

In [ ]:
!nvidia-smi

In [ ]:
import os, shutil, subprocess, sys
if os.path.isdir('/kaggle/working/code'):
    shutil.rmtree('/kaggle/working/code')
subprocess.run(
    ['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, '/kaggle/working/code'],
    check=True,
)
os.chdir('/kaggle/working/code')
sys.path.insert(0, '/kaggle/working/code')
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from pipeline.dataset import ensure_dataset

zip_path  = Path(INPUT_DATASET_DIR) / 'dataset.zip'
meta_path = Path(INPUT_DATASET_DIR) / 'dataset.meta.json'
if not zip_path.exists():
    raise FileNotFoundError(
        f'dataset.zip not found at {zip_path}.\n'
        f'Available under /kaggle/input/:\n  ' + '\n  '.join(sorted(os.listdir("/kaggle/input")))
    )
data_yaml = ensure_dataset(zip_path, meta_path, '/kaggle/working/dataset')

In [ ]:
from pipeline.train import run as train_run
run_dir = train_run(
    model='yolo11n',
    config='configs/smoke.yaml',
    drive_runs_dir='/kaggle/working/runs',
    local_runs_dir='/kaggle/working/runs_tmp',
    data_yaml=data_yaml,
    dataset_meta_path=meta_path,
    base_config='configs/base.yaml',
)
print('Smoke run:', run_dir)

In [ ]:
from pipeline.evaluate import run as eval_run
out = eval_run(run_dir, data_yaml=data_yaml, drive_runs_dir='/kaggle/working/runs')
print('test mAP50:    ', out['overall']['mAP50'])
print('test mAP50-95: ', out['overall']['mAP50_95'])
print('per_class count:', len(out['per_class']))   # should be 9 after the recent bug fix